# Notebook 7 – Binning & Discretization

## What is Binning?

Binning, also called discretization, is the process of converting a continuous numerical feature into a smaller number of meaningful groups or intervals.

For example, instead of representing age as:

18, 19, 20, 21, 22, 23...

we can create groups such as:

- Child
- Young Adult
- Adult
- Senior

### Why Binning?

Binning can be useful to:

- Simplify continuous numerical values
- Create meaningful categories
- Reduce the effect of small variations
- Capture non-linear relationships
- Improve interpretability
- Represent domain-specific ranges

However, binning can also result in information loss because different numerical values may be placed into the same category.

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("Titanic-Dataset.csv")

df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## Understanding the Continuous Feature

The Titanic dataset contains `Age`, which is a continuous numerical feature.

Instead of using only the exact age value, we can divide passengers into meaningful age groups.

Before creating bins, we first inspect the distribution and range of the `Age` feature.

In [3]:
df["Age"].describe()

count    714.000000
mean      29.699118
std       14.526497
min        0.420000
25%       20.125000
50%       28.000000
75%       38.000000
max       80.000000
Name: Age, dtype: float64

## 1. Equal-Width Binning

Equal-width binning divides the range of a numerical feature into intervals having approximately equal widths.

For example, if the age range is divided into four bins, each bin covers a similar numerical range.

### Feature Name
`Age_Equal_Width`

### Source Columns
- `Age`

### Logic
Divide the minimum-to-maximum age range into four equal-width intervals.

### Reason
To simplify the continuous age feature into a small number of groups.

### Business Meaning
In a general business dataset, equal-width binning could divide income into fixed numerical ranges.

### ML Relevance
It can help models represent broad non-linear patterns and can make numerical information easier to interpret.

### Leakage Check
The target variable is not used. In a production ML workflow, bin boundaries should be learned from training data only.

### Final Decision
**Needs Further Analysis**

In [4]:
df["Age_Equal_Width"] = pd.cut(
    df["Age"],
    bins=4,
    labels=["Group_1", "Group_2", "Group_3", "Group_4"]
)

df[["Age", "Age_Equal_Width"]].head(10)

,Age,Age_Equal_Width
0,22.0,Group_2
1,38.0,Group_2
2,26.0,Group_2
3,35.0,Group_2
4,35.0,Group_2
5,NaN,NaN
6,54.0,Group_3
7,2.0,Group_1
8,27.0,Group_2
9,14.0,Group_1


## 2. Equal-Frequency Binning

Equal-frequency binning creates groups containing approximately the same number of observations.

This approach is also called quantile-based binning.

### Feature Name
`Age_Equal_Frequency`

### Source Columns
- `Age`

### Logic
Divide the age values into four groups based on their rank so that each group contains approximately the same number of passengers.

### Reason
To avoid creating bins with very different numbers of observations.

### Business Meaning
In a customer dataset, customers could be divided into groups based on their spending rank.

### ML Relevance
Equal-frequency groups can provide balanced categories and may help represent non-linear patterns.

### Leakage Check
No target information is used. The bin boundaries should be learned using training data only.

### Final Decision
**Retain**

In [5]:
df["Age_Equal_Frequency"] = pd.qcut(
    df["Age"],
    q=4,
    labels=["Q1", "Q2", "Q3", "Q4"],
    duplicates="drop"
)

df[["Age", "Age_Equal_Frequency"]].head(10)

,Age,Age_Equal_Frequency
0,22.0,Q2
1,38.0,Q3
2,26.0,Q2
3,35.0,Q3
4,35.0,Q3
5,NaN,NaN
6,54.0,Q4
7,2.0,Q1
8,27.0,Q2
9,14.0,Q1


## 3. Quantile Binning

Quantile binning divides a numerical feature according to percentile boundaries.

For example:

- Q1 → lower values
- Q2 → lower-middle values
- Q3 → upper-middle values
- Q4 → higher values

Quantile binning is useful when the numerical feature is skewed because it can create groups with a more balanced number of observations.

### Feature Name
`Fare_Quantile_Group`

### Source Columns
- `Fare`

### Logic
Divide `Fare` into four quantile-based groups.

### Reason
Fare has a wide range and may contain highly different values. Quantile groups provide a simpler representation.

### Business Meaning
In an e-commerce dataset, customers could be grouped according to spending percentiles.

### ML Relevance
The feature can help a model identify different levels of fare without relying only on the exact numerical value.

### Leakage Check
The target is not used. Quantile boundaries should be calculated using training data only in an ML workflow.

### Final Decision
**Retain**

In [6]:
df["Fare_Quantile_Group"] = pd.qcut(
    df["Fare"],
    q=4,
    labels=["Low", "Medium", "High", "Very_High"],
    duplicates="drop"
)

df[["Fare", "Fare_Quantile_Group"]].head(10)

,Fare,Fare_Quantile_Group
0,7.2500,Low
1,71.2833,Very_High
2,7.9250,Medium
3,53.1000,Very_High
4,8.0500,Medium
5,8.4583,Medium
6,51.8625,Very_High
7,21.0750,High
8,11.1333,Medium
9,30.0708,High


## 4. Domain-Based Binning

Domain-based binning creates categories using meaningful business or domain rules rather than automatically calculated intervals.

For age, a common domain-based grouping is:

- Child: 0–12
- Teenager: 13–17
- Young Adult: 18–30
- Adult: 31–50
- Senior: 51+

### Feature Name
`Age_Group`

### Source Columns
- `Age`

### Logic
Assign each passenger to an age category based on predefined age ranges.

### Reason
The categories have an understandable real-world meaning.

### Business Meaning
Age groups can be used in many domains such as healthcare, insurance, marketing, and customer analysis.

### ML Relevance
The feature can help a model capture different behavioral patterns across age groups.

### Leakage Check
The target is not used. The ranges are predefined and therefore do not depend on the target.

### Final Decision
**Retain**

In [7]:
bins = [0, 12, 17, 30, 50, np.inf]
labels = ["Child", "Teenager", "Young Adult", "Adult", "Senior"]

df["Age_Group"] = pd.cut(
    df["Age"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

df[["Age", "Age_Group"]].head(10)

,Age,Age_Group
0,22.0,Young Adult
1,38.0,Adult
2,26.0,Young Adult
3,35.0,Adult
4,35.0,Adult
5,NaN,NaN
6,54.0,Senior
7,2.0,Child
8,27.0,Young Adult
9,14.0,Teenager


## 5. Continuous → Categorical

Binning converts a continuous numerical feature into categorical groups.

For example:

`Age = 28`

can become:

`Age_Group = Young Adult`

The original numerical information is still available in `Age`, while the new categorical feature provides a simplified representation.

### Advantage

It makes continuous values easier to interpret and can help represent non-linear relationships.

### Limitation

Binning loses some information because different values within the same bin become the same category.

For example, ages 20 and 30 may both belong to the same category even though their exact values are different.

In [8]:
df[["Age", "Age_Group"]].dropna().head(15)

,Age,Age_Group
0,22.0,Young Adult
1,38.0,Adult
2,26.0,Young Adult
3,35.0,Adult
4,35.0,Adult
6,54.0,Senior
7,2.0,Child
8,27.0,Young Adult
9,14.0,Teenager
10,4.0,Child


## 6. Risk Categories

Risk categories are commonly used in business, finance, insurance, and healthcare.

For example:

- Low Risk
- Medium Risk
- High Risk

The Titanic dataset does not contain an actual risk variable, so we can demonstrate the technique using `Fare` as a simple example.

### Feature Name
`Fare_Risk_Category`

### Source Columns
- `Fare`

### Logic
Use fare ranges to create Low, Medium, and High categories.

### Reason
To demonstrate how continuous numerical values can be converted into domain-style categories.

### Business Meaning
In a real banking or insurance dataset, similar categories could represent customer risk levels based on a validated risk score.

### ML Relevance
Risk categories can provide a simplified representation that some models can use as categorical information.

### Leakage Check
The target is not used. However, real risk categories must be defined without using future target outcomes.

### Final Decision
**Needs Further Analysis**

In [9]:
fare_bins = [-np.inf, 10, 50, np.inf]
fare_labels = ["Low", "Medium", "High"]

df["Fare_Risk_Category"] = pd.cut(
    df["Fare"],
    bins=fare_bins,
    labels=fare_labels
)

df[["Fare", "Fare_Risk_Category"]].head(10)

,Fare,Fare_Risk_Category
0,7.2500,Low
1,71.2833,High
2,7.9250,Low
3,53.1000,High
4,8.0500,Low
5,8.4583,Low
6,51.8625,High
7,21.0750,Medium
8,11.1333,Medium
9,30.0708,Medium


## 7. Income Groups

Income grouping is a common real-world example of binning.

For example:

- Low Income
- Middle Income
- High Income

The Titanic dataset does not contain an income column, so this concept is demonstrated using a small business example rather than pretending that Titanic contains income information.

### Feature Name
`Income_Group`

### Source Column
- `Income`

### Logic
Convert continuous income into predefined income ranges.

### Reason
Income groups are easier to interpret and can represent different customer segments.

### Business Meaning
Represents the customer's income segment.

### ML Relevance
Can help a model identify different behavioral patterns across income groups.

### Leakage Check
Income grouping itself does not require the target. The bin boundaries should be defined independently of future target information.

### Final Decision
**Retain**

In [10]:
income_example = pd.DataFrame({
    "Income": [18000, 30000, 45000, 65000, 90000, 150000]
})

income_bins = [0, 25000, 50000, 100000, np.inf]
income_labels = ["Low", "Lower_Middle", "Upper_Middle", "High"]

income_example["Income_Group"] = pd.cut(
    income_example["Income"],
    bins=income_bins,
    labels=income_labels
)

income_example

,Income,Income_Group
0,18000,Low
1,30000,Lower_Middle
2,45000,Lower_Middle
3,65000,Upper_Middle
4,90000,Upper_Middle
5,150000,High


## Advantages of Binning

Binning provides several advantages:

1. **Improves interpretability**  
   Categories are easier to understand than many individual numerical values.

2. **Captures non-linear relationships**  
   Different ranges can have different relationships with the target.

3. **Reduces sensitivity to small variations**  
   Small changes within the same bin may not change the category.

4. **Useful for domain-based analysis**  
   Business rules can create meaningful groups.

5. **Can simplify complex numerical data**  
   Continuous values can be represented using a smaller number of categories.

In [11]:
df["Age_Group"].value_counts(dropna=False)

Age_Group
Young Adult    296
Adult          241
NaN            177
Child           69
Senior          64
Teenager        44
Name: count, dtype: int64

## Limitations of Binning

Binning also has limitations:

1. **Information loss**  
   Exact numerical values are replaced by categories.

2. **Choice of bin boundaries matters**  
   Poorly selected boundaries can hide useful patterns.

3. **Too many bins increase complexity**  
   Too many categories may not provide meaningful simplification.

4. **Too few bins may oversimplify the data**  
   Important differences can be lost.

5. **Possible reduction in predictive performance**  
   Some models may perform better using the original continuous feature.

Therefore, binning should be evaluated rather than automatically applied to every numerical feature.

In [12]:
bin_comparison = df[
    ["Age", "Age_Equal_Width", "Age_Equal_Frequency", "Age_Group"]
].head(15)

bin_comparison

,Age,Age_Equal_Width,Age_Equal_Frequency,Age_Group
0,22.0,Group_2,Q2,Young Adult
1,38.0,Group_2,Q3,Adult
2,26.0,Group_2,Q2,Young Adult
3,35.0,Group_2,Q3,Adult
4,35.0,Group_2,Q3,Adult
5,NaN,NaN,NaN,NaN
6,54.0,Group_3,Q4,Senior
7,2.0,Group_1,Q1,Child
8,27.0,Group_2,Q2,Young Adult
9,14.0,Group_1,Q1,Teenager


## Equal-Width vs Equal-Frequency vs Quantile vs Domain-Based Binning

| Technique | How it works | Best Use |
|---|---|---|
| Equal-Width | Equal numerical interval widths | Simple numerical ranges |
| Equal-Frequency | Approximately equal observations per group | Balanced groups |
| Quantile | Uses percentile boundaries | Skewed distributions |
| Domain-Based | Uses predefined meaningful rules | Business/domain knowledge |

### Important Difference

Equal-frequency binning and quantile binning are closely related. Quantile binning specifically uses percentile/quantile boundaries to divide the data.

Domain-based binning is different because the boundaries come from domain knowledge rather than from the distribution of the dataset.

In [13]:
comparison = df[
    [
        "Age",
        "Age_Equal_Width",
        "Age_Equal_Frequency",
        "Age_Group"
    ]
].head(10)

comparison

,Age,Age_Equal_Width,Age_Equal_Frequency,Age_Group
0,22.0,Group_2,Q2,Young Adult
1,38.0,Group_2,Q3,Adult
2,26.0,Group_2,Q2,Young Adult
3,35.0,Group_2,Q3,Adult
4,35.0,Group_2,Q3,Adult
5,NaN,NaN,NaN,NaN
6,54.0,Group_3,Q4,Senior
7,2.0,Group_1,Q1,Child
8,27.0,Group_2,Q2,Young Adult
9,14.0,Group_1,Q1,Teenager


## Required Feature Documentation

### Feature: Age_Equal_Width

**Feature Name:** Age_Equal_Width  
**Source Columns:** Age  
**Logic:** Divide Age into four equal-width intervals.  
**Reason:** Simplify continuous age values.  
**Business Meaning:** Represents broad numerical age ranges.  
**ML Relevance:** Can capture broad non-linear age patterns.  
**Leakage Check:** No target used; bin boundaries should be learned from training data.  
**Final Decision:** Needs Further Analysis

### Feature: Age_Equal_Frequency

**Feature Name:** Age_Equal_Frequency  
**Source Columns:** Age  
**Logic:** Divide Age into four groups containing approximately equal numbers of observations.  
**Reason:** Create balanced age categories.  
**Business Meaning:** Represents relative age position.  
**ML Relevance:** Can represent non-linear patterns using categorical groups.  
**Leakage Check:** No target used; calculate boundaries from training data.  
**Final Decision:** Retain

### Feature: Fare_Quantile_Group

**Feature Name:** Fare_Quantile_Group  
**Source Columns:** Fare  
**Logic:** Divide Fare into four quantile groups.  
**Reason:** Represent different fare levels.  
**Business Meaning:** Represents relative fare segments.  
**ML Relevance:** Can help identify different fare-level patterns.  
**Leakage Check:** No target used; quantile boundaries should come from training data.  
**Final Decision:** Retain

### Feature: Age_Group

**Feature Name:** Age_Group  
**Source Columns:** Age  
**Logic:** Apply predefined age ranges.  
**Reason:** Create meaningful age categories.  
**Business Meaning:** Represents passenger age segments.  
**ML Relevance:** Helps capture different patterns across age groups.  
**Leakage Check:** No target used.  
**Final Decision:** Retain

### Feature: Fare_Risk_Category

**Feature Name:** Fare_Risk_Category  
**Source Columns:** Fare  
**Logic:** Divide Fare into Low, Medium, and High ranges.  
**Reason:** Demonstrate risk-style categorization.  
**Business Meaning:** Represents fare-level segments; it is not an actual Titanic risk measure.  
**ML Relevance:** Provides a categorical representation of fare.  
**Leakage Check:** No target used.  
**Final Decision:** Needs Further Analysis

### Feature: Income_Group

**Feature Name:** Income_Group  
**Source Columns:** Income  
**Logic:** Divide income into predefined ranges.  
**Reason:** Create meaningful customer segments.  
**Business Meaning:** Represents income level.  
**ML Relevance:** Can capture differences between income segments.  
**Leakage Check:** Should be defined without using target information.  
**Final Decision:** Retain

In [14]:
documentation = pd.DataFrame({
    "Feature Name": [
        "Age_Equal_Width",
        "Age_Equal_Frequency",
        "Fare_Quantile_Group",
        "Age_Group",
        "Fare_Risk_Category",
        "Income_Group"
    ],
    "Source Columns": [
        "Age",
        "Age",
        "Fare",
        "Age",
        "Fare",
        "Income"
    ],
    "Leakage Check": [
        "No target used",
        "No target used",
        "No target used",
        "No target used",
        "No target used",
        "No target used"
    ],
    "Final Decision": [
        "Needs Further Analysis",
        "Retain",
        "Retain",
        "Retain",
        "Needs Further Analysis",
        "Retain"
    ]
})

documentation

,Feature Name,Source Columns,Leakage Check,Final Decision
0,Age_Equal_Width,Age,No target used,Needs Further Analysis
1,Age_Equal_Frequency,Age,No target used,Retain
2,Fare_Quantile_Group,Fare,No target used,Retain
3,Age_Group,Age,No target used,Retain
4,Fare_Risk_Category,Fare,No target used,Needs Further Analysis
5,Income_Group,Income,No target used,Retain


## Final Conclusion

Binning and discretization convert continuous numerical features into meaningful groups.

In this notebook, we demonstrated:

- Equal-Width Binning
- Equal-Frequency Binning
- Quantile Binning
- Domain-Based Binning
- Continuous → Categorical conversion
- Age Groups
- Income Groups
- Risk Categories

Binning can improve interpretability and help represent non-linear relationships, but it can also cause information loss.

Therefore, binning should be applied only when the resulting categories have a meaningful purpose and provide value to the machine learning problem.

In [15]:
final_features = [
    "Age_Equal_Width",
    "Age_Equal_Frequency",
    "Fare_Quantile_Group",
    "Age_Group",
    "Fare_Risk_Category"
]

df[final_features].head(10)

,Age_Equal_Width,Age_Equal_Frequency,Fare_Quantile_Group,Age_Group,Fare_Risk_Category
0,Group_2,Q2,Low,Young Adult,Low
1,Group_2,Q3,Very_High,Adult,High
2,Group_2,Q2,Medium,Young Adult,Low
3,Group_2,Q3,Very_High,Adult,High
4,Group_2,Q3,Medium,Adult,Low
5,NaN,NaN,Medium,NaN,Low
6,Group_3,Q4,Very_High,Senior,High
7,Group_1,Q1,High,Child,Medium
8,Group_2,Q2,Medium,Young Adult,Medium
9,Group_1,Q1,High,Teenager,Medium
